# PySpark Data Transformation Pipeline
Update the configuration in Cell 2, then run the notebook from top to bottom. The pipeline reads a CSV, applies reusable cleansing and type-standardization rules, validates output quality, and writes Parquet.
from __future__ import annotations


In [1]:
import logging
import os
from pathlib import Path

logging.basicConfig(level=logging.INFO)
from pyspark.sql import functions as F

from common.spark import build_spark_session

MINIO_BUCKET = os.environ.get("MINIO_BUCKET", "data-bucket")
# Define directory paths
BRONZE_DIR = f"s3a://{MINIO_BUCKET}/bronze"
FIXTURES_DIR = Path.cwd() / "tests" / "fixtures"
SILVER_DIR = f"s3a://{MINIO_BUCKET}/silver"
GOLD_DIR = f"s3a://{MINIO_BUCKET}/gold"


## 1. Initialize PySpark Session


In [2]:
spark = build_spark_session()
print(f"Spark {spark.version} is ready with MinIO integration.")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 100)


:: loading settings :: url = jar:file:/Users/Anuj.Kumar.Gupta@telenor.com/data_end_to_end/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/Anuj.Kumar.Gupta@telenor.com/.ivy2/cache
The jars for the packages stored in: /Users/Anuj.Kumar.Gupta@telenor.com/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6d7925d3-227a-4ddd-9ef0-07974c296dda;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 81ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|  

Spark 3.5.5 is ready with MinIO integration.


In [9]:
from common.utils import (
    build_create_table_sql,
    load_dataframe,
    store_df_to_table,
    validate_required_columns,
    required_columns,
)


## 2. Load Source DataFrames


In [4]:
df = load_dataframe(spark, f"{BRONZE_DIR}/input.csv", input_format="csv")
df.show()
print(f"Rows: {df.count()}")
df.printSchema()


26/09/14 11:51:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+-----------+-------------+------+------+----------+
|customer_id|customer_name|region|amount|order_date|
+-----------+-------------+------+------+----------+
|        101|        Alice| North| 250.0|2024-01-05|
|        102|          Bob| South| 180.5|2024-01-08|
|        103|      Charlie|  East|320.75|2024-01-12|
|        104|        David|  West| 210.0|2024-01-15|
|        105|          Eve| North|410.25|2024-01-18|
|        106|        Frank| South| 195.4|2024-01-22|
|        107|        Grace|  East| 275.8|2024-01-25|
|        108|        Henry|  West| 330.1|2024-01-29|
|        109|          Ivy| North| 260.6|2024-02-02|
|        110|         Jack| South| 420.9|2024-02-05|
+-----------+-------------+------+------+----------+

Rows: 10
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: date (nullable = true)



## Cast dataframe columns, include metadata and descriptions
    spark.sql(f"DESCRIBE TABLE EXTENDED {table_name}").show(truncate=False)
Define all column metadata in a dictionary, then generate SQL dynamically.

In [ ]:



SILVER_TABLE_NAME = "silver.customer_orders"

# Column metadata: "source"/"type"/"is_required" drive the catalog table, the rest feeds the data dictionary.
SILVER_COLUMN_CONFIG = {
    "customer_id": {
        "source": "customer_id",
        "type": "INT",
        "is_required": True,
        "description": "Unique identifier for the customer who placed the order.",
        "valid_range": ">= 1, non-null",
    },
    "customer_name": {
        "source": "customer_name",
        "type": "STRING",
        "is_required": True,
        "description": "Customer's display name.",
    },
    "region": {
        "source": "region",
        "type": "STRING",
        "is_required": True,
        "description": "Sales region the order belongs to.",
    },
    "amount": {
        "source": "amount",
        "type": "DECIMAL(18,2)",
        "is_required": True,
        "description": "Order amount, cleaned and cast from the bronze source.",
        "unit": "USD (assumed \u2014 confirm currency with source system)",
        "valid_range": ">= 0",
    },
    "order_date": {
        "source": "order_date",
        "type": "DATE",
        "is_required": True,
        "description": "Date the order was placed.",
        "valid_range": "not in the future",
    },
}

SILVER_TABLE_METADATA = {
    "comment": "Cleaned, validated, and deduplicated customer order data (bronze to silver).",
    "grain": "One row per customer_id.",
    "primary_key": ["customer_id"],
    "freshness": (
        "Updated daily by the `customer-bronze-to-gold` Prefect deployment "
        "(cron 0 2 * * * UTC), ahead of the gold stage."
    ),
    "caveats": [
        "Despite covering 'orders', deduplication is keyed only on customer_id "
        "(clean_dataframe with source_key_columns=['customer_id']), so there is at most "
        "one row per customer, not one row per order.",
        "region enum values have not been confirmed against the source system; treat as free text.",
    ],
}

REQUIRED_COLUMNS = required_columns(SILVER_COLUMN_CONFIG)
validate_required_columns(df, REQUIRED_COLUMNS)


create_table_sql = build_create_table_sql(
    table_name=SILVER_TABLE_NAME,
    column_config=SILVER_COLUMN_CONFIG,
    location=f"{SILVER_DIR}/customer_orders",
    table_format="PARQUET",
    partition_columns=["region"],
    table_comment=SILVER_TABLE_METADATA["comment"],
)

store_df_to_table(spark,
    df=df,
    table_name=SILVER_TABLE_NAME,
    column_config=SILVER_COLUMN_CONFIG,
    create_table_sql=create_table_sql,
    write_mode="overwrite"
)

INFO:common.utils:All required columns are present: ['amount', 'customer_id', 'customer_name', 'order_date', 'region']


In [6]:
# Verify the inserted data.
spark.sql(f"""
    SELECT *
    FROM {SILVER_TABLE_NAME}
""")
# spark.sql(create_t

customer_id,customer_name,amount,order_date,region
101,Alice,250.00,2024-01-05,North
105,Eve,410.25,2024-01-18,North
109,Ivy,260.60,2024-02-02,North
102,Bob,180.50,2024-01-08,South
106,Frank,195.40,2024-01-22,South
110,Jack,420.90,2024-02-05,South
103,Charlie,320.75,2024-01-12,East
107,Grace,275.80,2024-01-25,East
104,David,210.00,2024-01-15,West
108,Henry,330.10,2024-01-29,West


In [7]:
spark.sql(
    f"DESCRIBE EXTENDED {SILVER_TABLE_NAME}"
).show(truncate=False)


+----------------------------+----------------------------------------------------------------------------+--------------------------------------------------------+
|col_name                    |data_type                                                                   |comment                                                 |
+----------------------------+----------------------------------------------------------------------------+--------------------------------------------------------+
|customer_id                 |int                                                                         |Unique identifier for the customer who placed the order.|
|customer_name               |string                                                                      |Customer's display name.                                |
|amount                      |decimal(18,2)                                                               |Order amount, cleaned and cast from the bronze source.  |
|order_dat

In [8]:
display(SILVER_TABLE_NAME)

'silver.customer_orders'

## 3. Inspect and Enforce Schemas


In [ ]:
# Define expected casts as needed for your source, for example:
EXPECTED_TYPES = {"customer_id": "string", "amount": "decimal(18,2)", "event_date": "date"}

typed_df = source_df
for column_name, target_type in EXPECTED_TYPES.items():
    if column_name in typed_df.columns:
        typed_df = typed_df.withColumn(column_name, F.col(column_name).cast(target_type))

typed_df.printSchema()


## 4. Handle Nulls, Duplicates, and Type Casting


In [ ]:
rows_before = typed_df.count()
string_columns = [field.name for field in typed_df.schema.fields if isinstance(field.dataType, StringType)]

clean_df = typed_df
for column_name in string_columns:
    clean_df = clean_df.withColumn(
        column_name,
        F.when(F.trim(F.col(column_name)) == "", F.lit(None))
        .otherwise(F.trim(F.col(column_name)))
    )

non_null_expression = [F.col(column_name).isNotNull() for column_name in clean_df.columns]
if non_null_expression:
    clean_df = clean_df.filter(F.coalesce(*non_null_expression))

deduplication_columns = SOURCE_KEY_COLUMNS or clean_df.columns
clean_df = clean_df.dropDuplicates(deduplication_columns)


## 5. Standardize and Derive Columns


In [ ]:
transformed_df = clean_df

# Optional domain rules; uncomment after setting names that exist in your source.
# transformed_df = transformed_df.withColumn(
#     "event_timestamp", F.to_timestamp("event_timestamp", "yyyy-MM-dd HH:mm:ss")
# )
# transformed_df = transformed_df.withColumn(
#     "amount", F.regexp_replace("amount", r"[^0-9.-]", "").cast("decimal(18,2)")
# )
# transformed_df = transformed_df.withColumn(
#     "status_normalized", F.lower(F.trim(F.regexp_replace("status", r"\s+", " ")))
# )
# transformed_df = transformed_df.withColumn(
#     "is_high_value", F.when(F.col("amount") >= 1000, True).otherwise(False)
# )


## 6. Filter, Join, and Reshape Data


In [ ]:
# Apply source-specific filters, joins, or reshape operations here.
# filtered_df = transformed_df.filter(F.col("status_normalized") == "active")
# enriched_df = filtered_df.join(lookup_df, on="customer_id", how="left")
# pivoted_df = enriched_df.groupBy("region").pivot("category").sum("amount")
# final_df = pivoted_df.select(F.col("region").alias("sales_region"), "*")
final_df = transformed_df


## 7. Aggregate Metrics with GroupBy


In [ ]:
# Example KPI pattern after choosing real grouping and metric columns:
# metrics_df = (
#     final_df.groupBy("region")
#     .agg(
#         F.count("*").alias("row_count"),
#         F.sum("amount").alias("total_amount"),
#         F.avg("amount").alias("average_amount"),
#         F.countDistinct("customer_id").alias("distinct_customers"),
#     )
#     .orderBy(F.desc("total_amount"))
# )
# metrics_df.show(truncate=False)


## 8. Apply Window-Based Transformations


In [ ]:
# Latest-record and running-total examples after setting real columns:
# latest_window = Window.partitionBy("customer_id").orderBy(F.col("event_timestamp").desc())
# latest_df = final_df.withColumn("record_rank", F.row_number().over(latest_window)).filter("record_rank = 1")
# running_window = Window.partitionBy("customer_id").orderBy("event_timestamp").rowsBetween(Window.unboundedPreceding, Window.currentRow)
# running_df = final_df.withColumn("running_amount", F.sum("amount").over(running_window))


## 9. Run Data Quality Checks


In [ ]:
rows_after = final_df.count()
null_count_expressions = [
    F.sum(F.col(column_name).isNull().cast("long")).alias(column_name)
    for column_name in final_df.columns
]
null_counts = final_df.agg(*null_count_expressions) if null_count_expressions else None

quality_report = {
    "rows_before": rows_before,
    "rows_after": rows_after,
    "duplicates_or_empty_rows_removed": rows_before - rows_after,
}
print(quality_report)
if null_counts is not None:
    null_counts.show(truncate=False)

if SOURCE_KEY_COLUMNS:
    duplicate_keys = (
        final_df.groupBy(*SOURCE_KEY_COLUMNS).count().filter(F.col("count") > 1).count()
    )
    assert duplicate_keys == 0, f"Found {duplicate_keys} duplicate business keys"
assert rows_after > 0, "No rows remain after transformation"


## 10. Write Transformed Data to Storage


In [ ]:
(
    final_df.write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)
print(f"Wrote {rows_after:,} rows to {Path(OUTPUT_PATH).resolve()}")

# To partition output, replace the writer above with:
# final_df.write.mode("overwrite").partitionBy("event_date").parquet(OUTPUT_PATH)


## Stop Spark

In [ ]:
spark.stop()
print("Spark session stopped")